# Quantized HF -> GGUF (Colab + Drive)

Run `hf_quantized_to_gguf.py` using Drive paths in Colab.

Flow:
1. Mount Drive
2. Configure Drive/session paths
3. Load `HF_TOKEN` from env or `.env`
4. Run conversion (direct or dequantize)
5. Verify GGUF output


In [ ]:
# Mount Google Drive when running in Colab
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=True)
    print('Drive mounted at /content/drive')
except Exception:
    print('Not running in Colab (or Drive mount unavailable); continuing without mount.')


Mounted at /content/drive
Drive mounted at /content/drive


In [ ]:
# configure paths
from pathlib import Path

# Resolve project root from mounted Drive
_candidates = [
    Path('/content/drive/MyDrive/training-embedding'),
    Path('/content/drive/My Drive/training-embedding'),
]
PROJECT_ROOT = next((p for p in _candidates if p.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('training-embedding not found under /content/drive')

# Input quantized HF folder (on Drive)
QUANTIZED_MODEL_DIR = PROJECT_ROOT / 'models' / 'turkish-gemma-9b-v01-4bit'

# Single GGUF output (on Drive)
GGUF_PATH = PROJECT_ROOT / 'ollama' / 'downloads' / 'Turkish-Gemma-9b-v0.1.from-hf-4bit.F16.gguf'

# Conversion options
CONVERSION_MODE = 'direct'   # 'direct' or 'dequantize'
OUTTYPE = 'f16'              # f16/f32/bf16
LLAMA_CPP_DIR = Path('/content/llama.cpp')

# Optional dequantize mode settings
# Keep intermediate on session disk to avoid filling Drive
DEQUANTIZED_MODEL_DIR = Path('/content/turkish-gemma-9b-v01-dequantized')
DEQUANTIZED_DTYPE = 'float16'
KEEP_DEQUANTIZED = False

# Optional Ollama registration (Colab runtime Ollama + Drive model store)
REGISTER_OLLAMA = False
OLLAMA_MODEL_NAME = 'turkish-gemma-v01-from-hf4bit'
OLLAMA_BIN = '/content/ollama-runtime/bin/ollama'
OLLAMA_MODELS_DIR = str(PROJECT_ROOT / 'ollama' / 'models')
OLLAMA_HOST = '127.0.0.1:11434'
OLLAMA_NUM_CTX = 2048

print('PROJECT_ROOT:', PROJECT_ROOT)
print('QUANTIZED_MODEL_DIR:', QUANTIZED_MODEL_DIR)
print('GGUF_PATH:', GGUF_PATH)
print('LLAMA_CPP_DIR:', LLAMA_CPP_DIR)
print('CONVERSION_MODE:', CONVERSION_MODE)


PROJECT_ROOT: /content/drive/MyDrive/training-embedding
QUANTIZED_MODEL_DIR: /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-v01-4bit
GGUF_PATH: /content/drive/MyDrive/training-embedding/ollama/downloads/Turkish-Gemma-9b-v0.1.from-hf-4bit.F16.gguf
LLAMA_CPP_DIR: /content/llama.cpp
CONVERSION_MODE: direct


In [10]:
# configure HF token
import os

def _read_dotenv_value(dotenv_path: Path, key: str) -> str:
    if not dotenv_path.exists():
        return ''
    for raw_line in dotenv_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        if k.startswith('export '):
            k = k[len('export '):].strip()
        if k == key:
            return v.strip().strip('\"').strip("'")
    return ''

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    hf_token = _read_dotenv_value(PROJECT_ROOT / '.env', 'HF_TOKEN')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN configured.')
else:
    print('HF_TOKEN not found. Continuing unauthenticated.')


HF_TOKEN configured.


In [13]:
!pip install -U transformers bitsandbytes accelerate sentencepiece safetensors


In [14]:
# run quantization
import os
import shlex
import subprocess
import sys

script_path = PROJECT_ROOT / 'hf_quantized_to_gguf.py'
if not script_path.exists():
    raise FileNotFoundError(f'Missing script: {script_path}')

cmd = [
    sys.executable, str(script_path),
    '--quantized-model-dir', str(QUANTIZED_MODEL_DIR),
    '--gguf-path', str(GGUF_PATH),
    '--llama-cpp-dir', str(LLAMA_CPP_DIR),
    '--outtype', OUTTYPE,
    '--conversion-mode', CONVERSION_MODE,
]

CONVERSION_MODE = "dequantize"
DEQUANTIZED_MODEL_DIR = Path("/content/turkish-gemma-9b-v01-dequantized")
KEEP_DEQUANTIZED = False

if REGISTER_OLLAMA:
    cmd += [
        '--register-ollama',
        '--ollama-model-name', OLLAMA_MODEL_NAME,
        '--ollama-bin', OLLAMA_BIN,
        '--ollama-host', OLLAMA_HOST,
        '--ollama-num-ctx', str(OLLAMA_NUM_CTX),
    ]
    if OLLAMA_MODELS_DIR:
        cmd += ['--ollama-models-dir', OLLAMA_MODELS_DIR]

env = os.environ.copy()
print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))
# subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)

p = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, text=True, capture_output=True)
print(p.stdout)
print(p.stderr)
print("returncode:", p.returncode)



Running command:
/usr/bin/python3 /content/drive/MyDrive/training-embedding/hf_quantized_to_gguf.py --quantized-model-dir /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-v01-4bit --gguf-path /content/drive/MyDrive/training-embedding/ollama/downloads/Turkish-Gemma-9b-v0.1.from-hf-4bit.F16.gguf --llama-cpp-dir /content/llama.cpp --outtype f16 --conversion-mode dequantize
Loading quantized model from: /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-v01-4bit

`torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 464/464 [00:36<00:00, 12.59it/s, Materializing param=model.norm.weight]
The modules are dequantized in torch.float16 and casted to torch.float16.

returncode: -9


In [16]:
def _size_gib(path: Path) -> float:
    return path.stat().st_size / (1024**3)

if GGUF_PATH.exists():
    print(f'{GGUF_PATH} -> {_size_gib(GGUF_PATH):.2f} GiB')
else:
    print(f'{GGUF_PATH} -> MISSING')


/content/drive/MyDrive/training-embedding/ollama/downloads/Turkish-Gemma-9b-v0.1.from-hf-4bit.F16.gguf -> MISSING


In [ ]:
# Optional cleanup
REMOVE_GGUF = False

if REMOVE_GGUF and GGUF_PATH.exists():
    GGUF_PATH.unlink()
    print(f'Removed GGUF: {GGUF_PATH}')
else:
    print('Cleanup skipped.')
